# Toolsmith Evening 1: raw llama.cpp serving

Run this notebook in a Colab L4 runtime. Make this repository available at `/content/bare-metal-agent` by uploading it or cloning the private repository with your own credential helper. Never paste credentials into notebook cells.

Before starting, add `HF_TOKEN` and `LOCAL_KEY` in Colab Secrets and enable notebook access. `LOCAL_KEY` protects the local llama.cpp server. Record the Colab units shown by the resource panel before and after the session; the notebook cannot read that value automatically.

In [ ]:
import json
import os
import platform
import shlex
import shutil
import subprocess
import time
from datetime import datetime, timezone
from pathlib import Path

gpu_result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'],
    check=True,
    capture_output=True,
    text=True,
)
gpu_info = gpu_result.stdout.strip()
print(gpu_info)
print(f'Python {platform.python_version()}')

In [ ]:
from google.colab import drive, userdata

PROJECT_DIR = Path('/content/bare-metal-agent')
if not (PROJECT_DIR / 'pyproject.toml').is_file():
    raise FileNotFoundError(
        'Upload or clone the repository to /content/bare-metal-agent before continuing.'
    )

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/toolsmith')
DRIVE_BUILD_CACHE = DRIVE_ROOT / 'builds'
DRIVE_MODEL_CACHE = DRIVE_ROOT / 'models'
DRIVE_RECORDS = DRIVE_ROOT / 'records'
for directory in (DRIVE_BUILD_CACHE, DRIVE_MODEL_CACHE, DRIVE_RECORDS):
    directory.mkdir(parents=True, exist_ok=True)

HF_TOKEN = userdata.get('HF_TOKEN')
LOCAL_KEY = userdata.get('LOCAL_KEY')
if not HF_TOKEN or not LOCAL_KEY:
    raise ValueError('HF_TOKEN and LOCAL_KEY must be set in Colab Secrets.')

In [ ]:
get_ipython().run_line_magic(
    'pip',
    f"install -q -e {shlex.quote(str(PROJECT_DIR) + '[colab]')}",
)

## Build or restore llama.cpp

The revision is pinned so that later parser fixtures remain reproducible. The build is cached on Drive, then copied to Colab's local disk before use. Change the revision deliberately and record any resulting protocol differences.

In [ ]:
LLAMA_CPP_REF = 'b5401'
LOCAL_SOURCE = Path('/content/llama.cpp-source')
LOCAL_BUILD = Path('/content/llama.cpp-build')
CACHED_BUILD = DRIVE_BUILD_CACHE / f'llama.cpp-{LLAMA_CPP_REF}-cuda'
SERVER_BIN = LOCAL_BUILD / 'bin' / 'llama-server'

if not SERVER_BIN.is_file() and (CACHED_BUILD / 'bin' / 'llama-server').is_file():
    shutil.copytree(CACHED_BUILD, LOCAL_BUILD, dirs_exist_ok=True)

if not SERVER_BIN.is_file():
    if not (LOCAL_SOURCE / '.git').exists():
        subprocess.run(
            ['git', 'clone', '--filter=blob:none', 'https://github.com/ggml-org/llama.cpp.git', str(LOCAL_SOURCE)],
            check=True,
        )
    subprocess.run(
        ['git', '-C', str(LOCAL_SOURCE), 'fetch', '--depth', '1', 'origin', 'tag', LLAMA_CPP_REF],
        check=True,
    )
    subprocess.run(
        ['git', '-C', str(LOCAL_SOURCE), 'checkout', '--detach', LLAMA_CPP_REF],
        check=True,
    )
    subprocess.run(
        [
            'cmake', '-S', str(LOCAL_SOURCE), '-B', str(LOCAL_BUILD),
            '-DGGML_CUDA=ON', '-DCMAKE_BUILD_TYPE=Release', '-DLLAMA_CURL=ON',
        ],
        check=True,
    )
    subprocess.run(
        ['cmake', '--build', str(LOCAL_BUILD), '--target', 'llama-server', '-j', '2'],
        check=True,
    )
    shutil.copytree(LOCAL_BUILD, CACHED_BUILD, dirs_exist_ok=True)

if not SERVER_BIN.is_file():
    raise FileNotFoundError(f'llama-server was not built at {SERVER_BIN}')

library_path = str(LOCAL_BUILD / 'lib')
os.environ['LD_LIBRARY_PATH'] = library_path + os.pathsep + os.environ.get('LD_LIBRARY_PATH', '')
print(f'Using llama.cpp {LLAMA_CPP_REF}: {SERVER_BIN}')

In [ ]:
from huggingface_hub import hf_hub_download

MODEL_REPO = 'Qwen/Qwen3-8B-GGUF'
MODEL_FILE = 'Qwen3-8B-Q4_K_M.gguf'
LOCAL_MODEL = Path('/content') / MODEL_FILE

cached_model = Path(
    hf_hub_download(
        repo_id=MODEL_REPO,
        filename=MODEL_FILE,
        cache_dir=str(DRIVE_MODEL_CACHE),
        token=HF_TOKEN,
    )
)
if not LOCAL_MODEL.exists() or LOCAL_MODEL.stat().st_size != cached_model.stat().st_size:
    shutil.copy2(cached_model, LOCAL_MODEL)

print(f'Local model: {LOCAL_MODEL} ({LOCAL_MODEL.stat().st_size:,} bytes)')

In [ ]:
from toolsmith.backends import (
    CompletionTruncatedError,
    LlamaCppClient,
    LlamaCppError,
)

SERVER_LOG_PATH = Path('/content/llama-server.log')
server_log_file = SERVER_LOG_PATH.open('w')
server_command = [
    str(SERVER_BIN),
    '-m', str(LOCAL_MODEL),
    '--jinja',
    '-ngl', '99',
    '-c', '16384',
    '--host', '127.0.0.1',
    '--port', '8080',
    '--api-key', LOCAL_KEY,
]
server_process = subprocess.Popen(
    server_command,
    cwd=str(LOCAL_BUILD / 'bin'),
    env=os.environ.copy(),
    stdout=server_log_file,
    stderr=subprocess.STDOUT,
)
print(f'Server PID: {server_process.pid}; log: {SERVER_LOG_PATH}')

In [ ]:
client = LlamaCppClient('http://127.0.0.1:8080', LOCAL_KEY, timeout=10.0)
health = None
last_error = None
deadline = time.monotonic() + 600

while time.monotonic() < deadline:
    try:
        health = client.health()
        break
    except LlamaCppError as error:
        last_error = error
        if server_process.poll() is not None:
            break
        time.sleep(2)

if health is None:
    server_log_file.flush()
    log_tail = '\n'.join(SERVER_LOG_PATH.read_text().splitlines()[-30:])
    raise RuntimeError(f'llama-server did not become healthy. Log tail:\n{log_tail}') from last_error

print(json.dumps(health, indent=2))

## Raw completion smoke test

This request intentionally uses `/completion`, not llama.cpp's chat endpoint. Toolsmith will own prompt rendering and tool-call parsing in later evenings.

In [ ]:
prompt = 'Name one primary color. Respond with one word only.\nAnswer:'
started = time.perf_counter()
try:
    response = client.complete(prompt, max_tokens=128, temperature=0.0)
    outcome = 'ok'
except CompletionTruncatedError as error:
    response = error.response
    outcome = 'truncated'
elapsed_seconds = time.perf_counter() - started

record = {
    'timestamp': datetime.now(timezone.utc).isoformat(),
    'llama_cpp_ref': LLAMA_CPP_REF,
    'model_repo': MODEL_REPO,
    'model_file': MODEL_FILE,
    'gpu': gpu_info,
    'python_version': platform.python_version(),
    'request': {
        'prompt': prompt,
        'max_tokens': 128,
        'temperature': 0.0,
        'stream': False,
    },
    'response': response,
    'outcome': outcome,
    'elapsed_seconds': elapsed_seconds,
}
record_path = DRIVE_RECORDS / 'evening-01-smoke.jsonl'
with record_path.open('a') as record_file:
    record_file.write(json.dumps(record) + '\n')

print(json.dumps(response, indent=2))
print(f'Elapsed: {elapsed_seconds:.2f}s; record: {record_path}')
if outcome != 'ok':
    raise RuntimeError('The smoke test was truncated; inspect the recorded response before continuing.')

In [ ]:
if 'client' in globals():
    client.close()
if 'server_process' in globals() and server_process.poll() is None:
    server_process.terminate()
    try:
        server_process.wait(timeout=20)
    except subprocess.TimeoutExpired:
        server_process.kill()
        server_process.wait()
if 'server_log_file' in globals():
    server_log_file.close()

print('Server stopped. Record Colab units used, then delete or disconnect the runtime.')